# 3. TF-IDF transformacija i podela podataka

### 3.1 Učitavanje biblioteka i pretprocesiranih podataka

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import scipy.sparse

df = pd.read_csv('data/preprocessed_news.csv')
print(f'Ucitano {len(df)} clanaka')
print(f'Kolone: {list(df.columns)}')
df.head()

### 3.2 Provera podataka pre transformacije

In [ ]:
# Provera nedostajucih vrednosti u final_text
print(f'Nedostajuci final_text: {df["final_text"].isnull().sum()}')
print(f'Prazni final_text: {(df["final_text"].astype(str).str.strip() == "").sum()}')

# Uklanjanje redova bez teksta
df = df.dropna(subset=['final_text'])
df = df[df['final_text'].astype(str).str.strip() != '']
print(f'\nBroj clanaka nakon ciscenja: {len(df)}')

# Distribucija klasa
print(f'\nDistribucija klasa:')
print(df['label' if 'label' in df.columns else 'word_count'].value_counts())

### 3.3 Podela na trening i test skup

In [ ]:
# Labele
y = df['label' if 'label' in df.columns else 'word_count'].values
texts = df['final_text'].astype(str).values

# Podela: 80% trening, 20% test
# stratify=y - obezbeduje istu distribuciju klasa u oba skupa
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Trening skup: {len(X_train_text)} clanaka ({len(X_train_text)/len(y)*100:.0f}%)')
print(f'Test skup:    {len(X_test_text)} clanaka ({len(X_test_text)/len(y)*100:.0f}%)')
print(f'\nDistribucija klasa u trening skupu:')
print(f'  Lazne:    {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')
print(f'  Istinite: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print(f'\nDistribucija klasa u test skupu:')
print(f'  Lazne:    {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)')
print(f'  Istinite: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')


### 3.4 TF-IDF vektorizacija

In [ ]:
# Kreiranje TF-IDF vektorizera
# max_features=10000 - ogranicava na 10000 najvaznijih reci
# ngram_range=(1,2) - koristi unigrame (jednu rec) i bigrame (par uzastopnih reci)
# max_df=0.95 - ignorise reci koje se pojavljuju u vise od 95% dokumenata
# min_df=2 - ignorise reci koje se pojavljuju u manje od 2 dokumenta

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=2
)

# VAZNO: fit samo na trening skupu da ne bi doslo do data leakage-a
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f'Dimenzije trening TF-IDF matrice: {X_train.shape}')
print(f'Dimenzije test TF-IDF matrice:    {X_test.shape}')
print(f'Broj feature-a (reci/bigrama): {X_train.shape[1]}')
print(f'Tip matrice: {type(X_train).__name__}')


### 3.5 Pregled najvažnijih TF-IDF feature-ova

In [ ]:
# Top feature-i po prosecnom TF-IDF score-u (na trening skupu)
feature_names = tfidf.get_feature_names_out()
mean_tfidf = X_train.mean(axis=0).A1

# Top 20 reci po TF-IDF
top_indices = mean_tfidf.argsort()[-20:][::-1]
print('Top 20 feature-a po prosecnom TF-IDF score-u (trening skup):')
for i in top_indices:
    print(f'  {feature_names[i]:30s} {mean_tfidf[i]:.4f}')


### 3.6 Čuvanje rezultata

In [ ]:
import joblib

# Cuvanje TF-IDF matrica i labela
scipy.sparse.save_npz('data/X_train.npz', X_train)
scipy.sparse.save_npz('data/X_test.npz', X_test)
np.save('data/y_train.npy', y_train)
np.save('data/y_test.npy', y_test)

# Cuvanje TF-IDF vektorizera
joblib.dump(tfidf, 'data/tfidf_vectorizer.joblib')
print('Sacuvano:')
print(f'  data/X_train.npz  - trening matrica {X_train.shape}')
print(f'  data/X_test.npz   - test matrica {X_test.shape}')
print(f'  data/y_train.npy  - trening labele ({len(y_train)})')
print(f'  data/y_test.npy   - test labele ({len(y_test)})')
print(f'  data/tfidf_vectorizer.joblib - TF-IDF model')